# Aula 74

## Closures em Python

Closures ocorrem quando funções internas, definidas dentro de outras funções, referenciam variáveis livre do seu escopo. Variáveis livres são as variáveis que não são definidas no escopo da função interna (são da função externa)

Se a função externa retornar apenas a referência da função interna, então o interpretador precisará atrelar quaisquer referências a variáveis livres que a função interna precisar para que ela possa ser executada corretamente.

São muito usados em programação funcional, decoradores de função e algoritmos em geral.

In [ ]:
def externa(a):
    # Enclosing
    def interna(b):
        return f"{a} {b}"
    
    return interna

incompleto = externa('Luiz')
completo = incompleto('Miranda')
print(completo)

Luiz Miranda


## Quando usar closures?

- Para manter estados simples sem usar classes
- Para criar fábricas (factories) de funções
- Para encapsular o código e esconder nomes importantes de escopos amplos
- Para usar funções de callback (onde algo é feito por etapas)
- Para decoradores de função em Python
- Para programação funcional e algoritmos em geral


In [3]:
from collections.abc import Callable
type Logger = Callable[[str],None]

def make_logger(name: str, color: str, icon: str = "…") -> Logger:
    def logger(log: str) -> None:
        log_name = f"{color}[{name.upper(): <7}] "
        print(f"{log_name} {icon} {log}\033[0m")

    return logger


debug = make_logger("debug", "\033[032m", icon="…")
info = make_logger("info", "\033[034m", icon="✔")
warning = make_logger("warning", "\033[033m", icon="⚠")
error = make_logger("error", "\033[031m", icon="✘")

print()
debug("Esse é um log de DEBUG")
info("Aconteceu alguma coisa no código")
warning("OPA!!! PRESTENÇÃO MINNINUUUUU...")
error("Te falei que ia dar ruim...")
debug("Esse é um log de DEBUG")
debug("Esse é um log de DEBUG")
print()


[DEBUG  ]  … Esse é um log de DEBUG
[INFO   ]  ✔ Aconteceu alguma coisa no código
[WARNING]  ⚠ OPA!!! PRESTENÇÃO MINNINUUUUU...
[ERROR  ]  ✘ Te falei que ia dar ruim...
[DEBUG  ]  … Esse é um log de DEBUG
[DEBUG  ]  … Esse é um log de DEBUG



In [6]:
from collections.abc import Callable

# Factory (fábrica de funções)
def make_multiplier(multiplier: float, /) -> Callable[[float], float]:
    def multiplier_times(multiplicand: float, /) -> float:
        return multiplicand * multiplier

    return multiplier_times


print("\nMultiplicadores")
times_two = make_multiplier(2)  # Função interna precisará lembrar do 2
times_three = make_multiplier(3)  # Nesse caso do 3

print("3 * 2 =   ", times_two(54))  # 3 * [2] = 6 - [2] lembrado
print("3 * 5 =   ", times_three(5))  # 5 * [3] = 15 - [3] lembrado


Multiplicadores
3 * 2 =    108
3 * 5 =    15


In [ ]:
from collections.abc import Callable # Type de funções

def make_less_than_checker(min_value:int) -> Callable[[int], bool]:
    def is_less_than(value: int) -> bool:
        return value < min_value
    return is_less_than

print("\nValidatores simples")
less_than_ten = make_less_than_checker(10)  # 10 precisa ser lembrado

print("30 < 10   ", less_than_ten(30))  # 30 é menor do que 10? False
print("9 < 10   ", less_than_ten(9))  # 9 é menor do que 10? True



Validatores simples
30 < 10    False
9 < 10    True


In [10]:
from collections.abc import Callable

def with_callback(value:str, callback: Callable[[str],str]) -> Callable[[],str]:
    def runner() -> str:
        print(f"Realizando alguma operação com o valor {value!r}")
        return callback(value)
    
    return runner

def my_callback(value: str) -> str:
    print(f"Valor {value!r} recebido no callback")
    return value + " (callback executed)"

print("\nCallback")

execute_operation = with_callback("## Exemplo ##", callback=my_callback)
result = execute_operation()
print(f"Callback:    {result!r}")



Callback
Realizando alguma operação com o valor '## Exemplo ##'
Valor '## Exemplo ##' recebido no callback
Callback:    '## Exemplo ## (callback executed)'


In [ ]:
from typing import Protocol

class Operation[**P, R](Protocol):
    def __call__(self, *args: P.args, **kwargs: P.kwargs) -> R: ...


def cacher[**P, R](callback: Operation[P, R]) -> Callable[P, R]:
    cached_params: dict[tuple[object, ...], R] = {}

    def closure(*args: P.args, **kwargs: P.kwargs) -> R:
        if args in cached_params:
            result = cached_params[args]
            print(f"Cacher found result {result!r}")
        else:
            result = callback(*args, **kwargs)
        cached_params[args] = result

        return result

    return closure


def operation(*args: str) -> list[str]:
    import time

    values: list[str] = []

    for arg in args:
        print(f"Fazendo algo complexo ou demorado com {arg!r}...")
        time.sleep(1)
        values.append(arg)

    return values


print("\nCacher")
operation_cached = cacher(operation)

op1 = operation_cached("a", "b", "c")
op2 = operation_cached("a", "b", "c")  # em cache

op4 = operation_cached("b", "b", "c")
op5 = operation_cached("b", "b", "c")  # em cache


Cacher
Fazendo algo complexo ou demorado com 'a'...
Fazendo algo complexo ou demorado com 'b'...
Fazendo algo complexo ou demorado com 'c'...
Cacher found result ['a', 'b', 'c']
Fazendo algo complexo ou demorado com 'b'...
Fazendo algo complexo ou demorado com 'b'...
Fazendo algo complexo ou demorado com 'c'...
Cacher found result ['b', 'b', 'c']


In [13]:
@cacher
def get_from_db(id: int, /) -> str:
    import time

    names = ["Luiz", "Maria", "Helena", "Letícia"]

    print(f"Returning value for ID {id}")
    time.sleep(2)
    return names[id]


print("\nCacher Decorator")

print(get_from_db(1))
print(get_from_db(1))
print(get_from_db(0))
print(get_from_db(2))
print(get_from_db(0))
print(get_from_db(2))
print(get_from_db(0))
print(get_from_db(2))
print(get_from_db(0))
print(get_from_db(2))


Cacher Decorator
Returning value for ID 1
Maria
Cacher found result 'Maria'
Maria
Returning value for ID 0
Luiz
Returning value for ID 2
Helena
Cacher found result 'Luiz'
Luiz
Cacher found result 'Helena'
Helena
Cacher found result 'Luiz'
Luiz
Cacher found result 'Helena'
Helena
Cacher found result 'Luiz'
Luiz
Cacher found result 'Helena'
Helena


In [14]:
from collections.abc import Callable


def enclosing(a: str) -> Callable[[str], str]:
    def closure(b: str) -> str:
        return f"{a} {b}"

    return closure


same_closure = enclosing("João")  # aqui está o closure
this_is_result_str = same_closure("Otávio")  # isso é uma str

print()
print("enclosing")
# Como ver dados da função externa
# Variáveis exclusivamente locais
print("Varnames [Local    ]  ", enclosing.__code__.co_varnames)
# Variáveis do enclosing usadas nessa função
print("Freevars [Enclosing]  ", enclosing.__code__.co_freevars)
# Variáveis dessa função usadas em funções internas
print("Cellvars [Usadas   ]  ", enclosing.__code__.co_cellvars)
# Células da closure se existir
print("Closure  [Closure  ]  ", enclosing.__closure__)

print()
print()
print("same_closure")
# Como ver dados da função interna
# Variáveis exclusivamente locais
print("Varnames [Local    ]  ", same_closure.__code__.co_varnames)
# Variáveis do enclosing usadas nessa função
print("Freevars [Enclosing]  ", same_closure.__code__.co_freevars)
# Variáveis dessa função usadas em funções internas
print("Cellvars [Usadas   ]  ", same_closure.__code__.co_cellvars)
# Células da closure se existir
print("Closure  [Closure  ]  ", same_closure.__closure__[0].cell_contents)


enclosing
Varnames [Local    ]   ('a', 'closure')
Freevars [Enclosing]   ()
Cellvars [Usadas   ]   ('a',)
Closure  [Closure  ]   None


same_closure
Varnames [Local    ]   ('b',)
Freevars [Enclosing]   ('a',)
Cellvars [Usadas   ]   ()
Closure  [Closure  ]   João
